# Valid data in Bronze zone

In [1]:
from pyspark.sql import SparkSession, DataFrame
import os
import pyspark.sql.functions as F
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [2]:
SCHEMA_STORE = 'warehouse'

spark = (
    SparkSession.builder
    .appName('valid data in bronze zone')
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # jar files supporting read/write postgres, trino, S3, hive (delta table for lake house)
    .config(
        'spark.jars',
        """
            ../../../../spark/jar/trino-jdbc-480.jar,
            ../../../../spark/jar/postgresql-42.7.11.jar,
            ../../../../spark/jar/hadoop-aws-3.4.2.jar,
            ../../../../spark/jar/bundle-2.29.52.jar
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083")
    .config("spark.sql.catalogImplementation", "hive")
    # create schema using spark (instead of using trino)
    # .config("spark.sql.warehouse.dir", f"s3a://flezi-data/{SCHEMA_STORE}/")
    .enableHiveSupport()
    .getOrCreate()
)
spark

26/06/22 15:47:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
df_origin = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)
# df_origin.show(3)

In [3]:
# Connection variables
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""
target_table = "datalake.taxi_data.nyc_taxi"

# Read table into a DataFrame
df_bronze = (
    spark.read
    .option("pushDownPredicate", "false")
    .format("jdbc")
    .option("url", trino_url)
    .option("driver", "io.trino.jdbc.TrinoDriver")
    .option("dbtable", target_table)
    .option("user", trino_user)
    .option("password", trino_password)
    .load()
)
df_bronze.count()

7781765

In [ ]:
def validate_bronze(df_bronze:DataFrame, df_origin:DataFrame, valid_date:str):
    count_bronze = df_bronze.filter(
        df_bronze['pickup_datetime'] < valid_date
    ).count()

    count_origin = df_origin.filter(
        df_origin['pickup_datetime'] < valid_date
    ).count()

    print(count_bronze, count_origin)
    return count_bronze == count_origin

In [7]:
valid_date = '2009-02-02'
validate_bronze(df_bronze, df_origin, valid_date)

144619 144619


True

In [8]:
valid_date = '2009-03-01'
validate_bronze(df_bronze, df_origin, valid_date)

280322 280325


False